In [ ]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [ ]:
%run ../setup/00_setup_parameters 

In [ ]:
from pyspark.sql import functions as F

def process_silver_enriched_executions(microBatchDF, batchId):
    # 1. Register the micro-batch dataframe as a temp view named 'bronze_view'
    # This matches what your SQL files expect!
    microBatchDF.createOrReplaceTempView("bronze_view")
    
    # 2. Read the external SQL files from disk
    with open("silver_reject.sql", "r") as f:
        reject_sql_template = f.read()
        
    with open("silver_merge.sql", "r") as f:
        merge_sql_template = f.read()
        
    # 3. Inject Python variables ({catalog}, {schema}) into the SQL templates
    reject_sql = reject_sql_template.format(catalog=catalog, schema=schema)
    merge_sql = merge_sql_template.format(catalog=catalog, schema=schema)
    
    # 4. Execute the Reject and Merge operations against the micro-batch view
    microBatchDF.sparkSession.sql(reject_sql)
    microBatchDF.sparkSession.sql(merge_sql)

# 5. Start the stream reading all records from the Bronze table
query = (
    spark.readStream
    .table(f"{catalog}.{schema}.bronze_streaming_source")
    .writeStream
    .foreachBatch(process_silver_enriched_executions)
    .option("checkpointLocation", f"{landing_volume}/checkpoints/silver_enriched_executions")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()